# **T50销量榜单代码**

In [18]:
import requests
from bs4 import BeautifulSoup
import re
import time
import pandas as pd
import os

# ===================== 项目路径配置 =====================
PROJECT_ROOT = r"C:\Users\马琳琳\Desktop\数据分析与决策\2zuoye"
RAW_DATA_PATH = os.path.join(PROJECT_ROOT, "data_raw", "dangdang_python_books_raw.csv")

# ===================== 请求头配置 =====================
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Referer": "https://search.dangdang.com/"
}

# ===================== 工具函数 =====================
def extract_publish_year(text):
    """提取出版年份（4位数字）"""
    year_match = re.search(r"(\d{4})", text)
    return year_match.group(1) if year_match else ""

def extract_comment_num(comment_text):
    """提取纯数字评论数"""
    if not comment_text:
        return "0"
    num_match = re.search(r"(\d+)", comment_text)
    return num_match.group(1) if num_match else "0"

def calculate_discount(original, discount):
    """计算折扣（返回X.X折格式）"""
    try:
        ori_price = float(re.search(r"¥(\d+\.?\d*)", original).group(1))
        dis_price = float(re.search(r"¥(\d+\.?\d*)", discount).group(1))
        discount_rate = round((dis_price / ori_price) * 10, 1)
        return f"{discount_rate}折"
    except:
        return "无折扣"

def extract_price_from_text(price_text):
    """从价格文本中提取带¥的价格字符串（如¥75.80）"""
    if not price_text:
        return ""
    price_match = re.search(r"¥\d+\.?\d*", price_text)
    return price_match.group() if price_match else ""

def is_pure_digit(text):
    """判断是否为纯数字（如ISBN码/日期）"""
    if not text:
        return True
    return text.strip().isdigit()

def has_qizi_in_discount_price(price_text):
    """判断折后价（span.search_now_price）是否包含'起'字"""
    if not price_text:
        return False
    return "起" in str(price_text)

def is_invalid_author(author_str):
    """判断作者是否为无效值（加入购物车收藏/加入购物车购买电子书收藏）"""
    invalid_authors = ["加入购物车收藏", "加入购物车购买电子书收藏"]
    return author_str.strip() in invalid_authors

def get_detail_info(product_url):
    """访问详情页补全出版社、年份、简介、原价、折后价、作者（兜底）"""
    publisher = ""
    publish_year = ""
    intro = ""
    original_price_detail = ""  # 详情页原价
    discount_price_detail = ""  # 详情页折后价
    head_title = ""
    author_detail = ""  # 详情页作者（兜底）
    
    try:
        if not product_url.startswith("http"):
            product_url = "https:" + product_url
        response = requests.get(product_url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        # 1. 提取详情页作者（兜底：span#author.t1）
        author_tag = soup.select_one("span#author.t1")
        if author_tag:
            author_detail = author_tag.get_text(strip=True).replace("作者：", "").strip()
        
        # 2. 提取出版社
        pub_tag = soup.select_one('p dd[name="出版社"] + span a') or soup.select_one('p#publisher a')
        if pub_tag:
            publisher = pub_tag.get_text(strip=True)
        
        # 3. 提取出版年份
        time_tag = soup.find("p", string=re.compile(r"出版时间|出版日期"))
        if time_tag:
            publish_year = extract_publish_year(time_tag.get_text())
        
        # 4. 提取详情页原价（优先级：div#original-price.price_m → p.paper_price）
        original_price_tag1 = soup.select_one("div#original-price.price_m")
        if original_price_tag1:
            original_price_detail = extract_price_from_text(original_price_tag1.get_text(strip=True))
        # 兜底：p.paper_price
        if not original_price_detail:
            paper_price_tag = soup.select_one("p.paper_price")
            if paper_price_tag:
                original_price_detail = extract_price_from_text(paper_price_tag.get_text(strip=True))
        
        # 5. 提取详情页折后价：p#dd-price
        discount_price_tag = soup.select_one("p#dd-price")
        if discount_price_tag:
            discount_price_detail = extract_price_from_text(discount_price_tag.get_text(strip=True))
        
        # 6. 提取简介（多来源兜底）
        detail_tag = soup.select_one("p.detail")
        if detail_tag:
            intro = detail_tag.get_text(strip=True)
        
        if is_pure_digit(intro):
            head_title_tag = soup.select_one("span.head_title_name")
            if head_title_tag:
                head_title = head_title_tag.get_text(strip=True)
                intro = head_title
        
        if is_pure_digit(intro):
            new_edit_tag = soup.select_one("div#newEditModule.newEdit_box")
            if new_edit_tag:
                intro = new_edit_tag.get_text(strip=True).replace("\n", "").replace("\r", "")
        
        time.sleep(1)
    except Exception as e:
        print(f"⚠️ 详情页获取失败: {str(e)[:50]}")
    return publisher, publish_year, intro, original_price_detail, discount_price_detail, author_detail

# ===================== 核心爬虫函数 =====================
def crawl_dangdang_books():
    os.makedirs(os.path.dirname(RAW_DATA_PATH), exist_ok=True)
    
    all_books = []  # 最终有效数据
    filtered_price_books = []  # 折后价含"起"字被过滤的书籍
    filtered_author_books = []  # 作者无效被过滤的书籍
    current_page = 1
    TOTAL_TARGET = 50  # 最终保留前50条有效数据
    PAGE_SIZE = 20
    
    print("===== 开始爬取当当网Python书籍（最终保留有效数据前50条）=====")
    print("🔍 过滤规则1：折后价（span.search_now_price）含'起'字的商品")
    print("🔍 过滤规则2：作者为'加入购物车收藏'/'加入购物车购买电子书收藏'的商品")
    
    while len(all_books) < TOTAL_TARGET and current_page <= (TOTAL_TARGET // PAGE_SIZE + 5):  # 多爬5页备用
        params = {
            "key": "python",
            "act": "input",
            "sort_type": "sort_sale_amt_desc",
            "page_index": current_page
        }
        
        try:
            response = requests.get("https://search.dangdang.com/", headers=HEADERS, params=params, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")
            book_items = soup.select("ul.bigimg > li")
            
            for item in book_items:
                # 1. 书名 + 详情页链接
                book_name_tag = item.select_one("p.name > a")
                book_name = book_name_tag["title"].strip() if book_name_tag else "未知书籍"
                product_url = book_name_tag["href"] if book_name_tag else ""
                
                # 过滤规则1：折后价含"起"字
                discount_price_tag = item.select_one("span.search_now_price")
                discount_price = discount_price_tag.get_text(strip=True) if discount_price_tag else ""
                if has_qizi_in_discount_price(discount_price):
                    filtered_price_books.append(book_name)
                    print(f"🚫 过滤（折后价含起）：{book_name} | 折后价：{discount_price}")
                    continue
                
                # 2. 作者/出版年份/出版社（搜索页初步提取）
                author_pub_tag = item.select_one("p.search_book_author")
                author, publish_year, publisher = "", "", ""
                if author_pub_tag:
                    pub_text = author_pub_tag.get_text(strip=True)
                    parts = re.split(r"\s*/\s*", pub_text)
                    if len(parts) >= 1: author = parts[0]
                    if len(parts) >= 2: publish_year = extract_publish_year(parts[1])
                    if len(parts) >= 3: publisher = parts[2]
                
                # 强制替换指定书籍作者
                if "Python网络爬虫项目开发全程实录" in book_name:
                    author = "明日科技"
                    print(f"✅ 修正作者：{book_name} → 明日科技")
                
                # 过滤规则2：作者无效直接跳过
                if is_invalid_author(author):
                    filtered_author_books.append(book_name)
                    print(f"🚫 过滤（作者无效）：{book_name} | 作者：{author}")
                    continue
                
                # 3. 评论数
                comment_tag = item.select_one("a.search_comment_num")
                comment_num = extract_comment_num(comment_tag.get_text(strip=True)) if comment_tag else "0"
                
                # 4. 原价（搜索页）
                original_price_tag = item.select_one("span.search_pre_price")
                original_price = original_price_tag.get_text(strip=True) if original_price_tag else ""
                
                # 5. 搜索页简介
                intro_tag = item.select_one("p.detail")
                intro = intro_tag.get_text(strip=True) if intro_tag else ""
                
                # 6. 补全缺失字段
                original_price_detail = ""
                discount_price_detail = ""
                author_detail = ""
                if not publisher or not publish_year or is_pure_digit(intro) or not original_price or not discount_price or is_pure_digit(author):
                    if product_url:
                        print(f"🔍 补全详情页：{book_name[:20]}...")
                        detail_pub, detail_year, detail_intro, detail_ori_price, detail_dis_price, detail_author = get_detail_info(product_url)
                        publisher = publisher or detail_pub
                        publish_year = publish_year or detail_year
                        intro = intro if not is_pure_digit(intro) else detail_intro
                        original_price_detail = detail_ori_price
                        discount_price_detail = detail_dis_price
                        # 作者兜底（指定书籍除外）
                        if not "Python网络爬虫项目开发全程实录" in book_name:
                            author = author if not is_pure_digit(author) else detail_author
                
                # 最终价格
                final_original_price = original_price or original_price_detail
                final_discount_price = discount_price or discount_price_detail
                
                # 计算折扣
                discount = calculate_discount(final_original_price, final_discount_price)
                
                # 最终简介
                final_intro = intro if not is_pure_digit(intro) else "无简介"
                
                # 组装有效数据
                book_data = {
                    "书名": book_name,
                    "作者": author,
                    "出版年份": publish_year,
                    "出版社": publisher,
                    "评论数": comment_num,
                    "原价": final_original_price,
                    "折后价": final_discount_price,
                    "折扣": discount,
                    "简介": final_intro[:500],
                    "销量排名": len(all_books) + 1  # 直接分配最终排名
                }
                all_books.append(book_data)
                print(f"✅ 已采集有效数据 {len(all_books)}/{TOTAL_TARGET}：{book_name[:20]}...")
                
                # 达到目标条数则停止
                if len(all_books) >= TOTAL_TARGET:
                    break
            
            print(f"📄 第{current_page}页采集完成，有效数据累计{len(all_books)}条")
            current_page += 1
            time.sleep(2)
        
        except Exception as e:
            print(f"❌ 第{current_page}页采集失败: {str(e)[:50]}")
            current_page += 1
            continue
    
    # 最终仅保留前50条有效数据（兜底处理）
    final_books = all_books[:TOTAL_TARGET]
    
    # 保存原始数据
    df = pd.DataFrame(final_books)
    df.to_csv(RAW_DATA_PATH, index=False, encoding="utf-8-sig")
    
    # 输出统计信息
    print("\n===== 爬取&过滤完成！=====")
    print(f"🎯 最终有效数据：{len(final_books)}条（目标前50条）")
    print(f"💾 数据保存至：{RAW_DATA_PATH}")
    
    print("\n===== 过滤统计 =====")
    print(f"1. 折后价含'起'字被过滤：{len(filtered_price_books)}条")
    if filtered_price_books:
        for i, book in enumerate(filtered_price_books, 1):
            print(f"   {i}. {book}")
    
    print(f"\n2. 作者无效被过滤：{len(filtered_author_books)}条")
    if filtered_author_books:
        for i, book in enumerate(filtered_author_books, 1):
            print(f"   {i}. {book}")
    
    # 字段缺失统计
    empty_original = df[df["原价"] == ""].shape[0]
    empty_discount = df[df["折后价"] == ""].shape[0]
    empty_intro = df[df["简介"] == "无简介"].shape[0]
    empty_author = df[df["作者"] == ""].shape[0]
    print(f"\n===== 字段缺失统计 =====")
    print(f"作者缺失：{empty_author}条")
    print(f"出版社缺失：{df['出版社'].isnull().sum() + (df['出版社'] == '').sum()}条")
    print(f"出版年份缺失：{df['出版年份'].isnull().sum() + (df['出版年份'] == '').sum()}条")
    print(f"简介为空/纯数字：{empty_intro}条")
    print(f"原价缺失：{empty_original}条")
    print(f"折后价缺失：{empty_discount}条")
    
    return df, filtered_price_books, filtered_author_books

# ===================== 主程序执行 =====================
if __name__ == "__main__":
    df, filtered_price, filtered_author = crawl_dangdang_books()

===== 开始爬取当当网Python书籍（最终保留有效数据前50条）=====
🔍 过滤规则1：折后价（span.search_now_price）含'起'字的商品
🔍 过滤规则2：作者为'加入购物车收藏'/'加入购物车购买电子书收藏'的商品
✅ 已采集有效数据 1/50：Python编程从入门到实践 第3版 P...
🔍 补全详情页：Python网络爬虫与数据分析从入门到实...
✅ 已采集有效数据 2/50：Python网络爬虫与数据分析从入门到实...
🔍 补全详情页：Python完全自学教程...
✅ 已采集有效数据 3/50：Python完全自学教程...
🔍 补全详情页：深度学习的数学――使用Python语言...
✅ 已采集有效数据 4/50：深度学习的数学――使用Python语言...
🔍 补全详情页：Python+Office:轻松实现Py...
✅ 已采集有效数据 5/50：Python+Office:轻松实现Py...
🚫 过滤（作者无效）：当当网正版 图解C++开发基础图解Python开发基础图解Java开发基础 | 作者：加入购物车收藏
✅ 已采集有效数据 6/50：Python自动化办公很简单 告别烦琐的...
🔍 补全详情页：Python编程:从入门到实践(第3版)...
✅ 已采集有效数据 7/50：Python编程:从入门到实践(第3版)...
✅ 已采集有效数据 8/50：深度学习入门 基于Python的理论与实...
✅ 已采集有效数据 9/50：Python区块链量化交易 从区块链基础...
✅ 已采集有效数据 10/50：笨办法学Python 原书第5版 通过习...
🔍 补全详情页：你好!Python...
✅ 已采集有效数据 11/50：你好!Python...
✅ 已采集有效数据 12/50：小学生Python创意编程 视频教学版 ...
🚫 过滤（作者无效）：Python编程三剑客第3版：Python编程从入门到实践第3版+快速上手第2版+极客项目编程（当当套装共3册） | 作者：加入购物车购买电子书收藏
✅ 已采集有效数据 13/50：AI+Python医学数据分析实践 余本...
✅ 已采集有效数据 14/50：机器学习原理与Python实践 CSDN...
🔍 补全详情页：程序是怎样跑起来的(第3版)...
